In [12]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
from PIL import Image
import pandas as pd
import timm
import kagglehub

In [13]:
# 1. Download dataset from Kaggle

print("Downloading dataset...")

path = kagglehub.dataset_download(
    "paultimothymooney/chest-xray-pneumonia"
)

print("Dataset path:", path)



Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Dataset path: /kaggle/input/chest-xray-pneumonia


In [14]:
# 2. Prepare dataset (build DataFrame)

def prepare_data(base_path):
    data = []

    for split in ["train", "val", "test"]:
        for label_name in ["NORMAL", "PNEUMONIA"]:

            label = 0 if label_name == "NORMAL" else 1
            folder = os.path.join(base_path, split, label_name)

            if os.path.exists(folder):
                for img_name in os.listdir(folder):
                    if img_name.endswith((".jpg", ".jpeg", ".png")):
                        data.append({
                            "img_path": os.path.join(folder, img_name),
                            "label": label,
                            "split": split
                        })

    return pd.DataFrame(data)


df_all = prepare_data(os.path.join(path, "chest_xray"))

train_df = df_all[df_all["split"] == "train"]
val_df = df_all[df_all["split"] == "val"]
test_df = df_all[df_all["split"] == "test"]

print(
    f"Train: {len(train_df)} | "
    f"Val: {len(val_df)} | "
    f"Test: {len(test_df)}"
)

Train: 5216 | Val: 16 | Test: 624


In [15]:
# 3. Device + transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VIT_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((VIT_SIZE, VIT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [16]:
# 4. Dataset class

class XRayDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]["img_path"]
        image = Image.open(img_path).convert("RGB")

        label = torch.tensor(
            self.df.iloc[idx]["label"],
            dtype=torch.long
        )

        if self.transform:
            image = self.transform(image)

        return image, label


In [17]:
# 5. DataLoaders

train_loader = DataLoader(
    XRayDataset(train_df, transform),
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    XRayDataset(val_df, transform),
    batch_size=16
)


In [18]:
# 6. Vision Transformer model

print("Loading ViT model...")

model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=True,
    num_classes=2
).to(DEVICE)


optimizer = optim.Adam(model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()

Loading ViT model...


In [19]:
# 7. Training loop (simple test run)

print("Starting training...")

model.train()

for epoch in range(1):

    for i, (images, labels) in enumerate(train_loader):

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        if i % 10 == 0:
            print(f"Batch {i} | Loss: {loss.item():.4f}")

        if i == 50:
            break


print("Training completed successfully")

Starting training...
Batch 0 | Loss: 1.0496
Batch 10 | Loss: 0.3249
Batch 20 | Loss: 0.0296
Batch 30 | Loss: 0.5535
Batch 40 | Loss: 0.0306
Batch 50 | Loss: 0.0234
Training completed successfully
